In [1]:
!pip install transformers datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    get_linear_schedule_with_warmup
)
from datasets import load_dataset
import evaluate
from torch.optim import AdamW
import numpy as np
from tqdm import tqdm

# Kiểm tra GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Cấu hình
CONFIG = {
    "model_name": "t5-base",   # backbone PLM
    "adapter_dim": 64,              # r: bottleneck dimension (thay đổi để thử nghiệm)
    "num_labels": 2,                # QNLI: 2 nhãn (entailment / not_entailment)
    "max_length": 128,
    "batch_size": 32,
    "num_epochs": 5,
    "learning_rate": 1e-4,          # adapter dùng lr cao hơn full fine-tuning
    "warmup_ratio": 0.06,
    "weight_decay": 0.01,
    "seed": 42,
}

torch.manual_seed(CONFIG["seed"])

Using device: cuda


Định nghĩa Adapter Module

In [3]:
class AdapterModule(nn.Module):
    """
    Adapter module theo paper (Houlsby et al. 2019):
    h <- f(h W_down) W_up + h
    """
    def __init__(self, hidden_dim: int, adapter_dim: int):
        super().__init__()
        self.down_proj  = nn.Linear(hidden_dim, adapter_dim)
        self.up_proj    = nn.Linear(adapter_dim, hidden_dim)
        self.activation = nn.ReLU()

        # Khởi tạo gần 0 để adapter ban đầu ~ identity
        nn.init.normal_(self.down_proj.weight, std=1e-3)
        nn.init.zeros_(self.down_proj.bias)
        nn.init.normal_(self.up_proj.weight, std=1e-3)
        nn.init.zeros_(self.up_proj.bias)

    def forward(self, x):
        residual = x
        x = self.down_proj(x)
        x = self.activation(x)
        x = self.up_proj(x)
        return x + residual  # residual connection


class T5BlockWithAdapter(nn.Module):
    """
    Bọc T5Block gốc và chèn adapter bằng forward hooks.
    - adapter_attn: chèn sau Self-Attention sub-layer (layer[0])
    - adapter_ffn : chèn sau FFN sub-layer (layer[-1])
    Cách này không đụng vào internal API của T5Block nên tránh được
    mọi vấn đề về signature thay đổi giữa các phiên bản transformers.
    """
    def __init__(self, original_block, hidden_dim: int, adapter_dim: int):
        super().__init__()
        self.block = original_block
        self.adapter_attn = AdapterModule(hidden_dim, adapter_dim)
        self.adapter_ffn  = AdapterModule(hidden_dim, adapter_dim)

        # Lưu trữ hidden_states trung gian qua hook
        self._attn_out = None
        self._ffn_out  = None

        # Hook sau Self-Attention: layer[0] trả về tuple (hidden_states, ...)
        def hook_attn(module, input, output):
            h = output[0]
            h = self.adapter_attn(h)
            # Trả về tuple với hidden_states đã được adapter xử lý
            return (h,) + output[1:]

        # Hook sau FFN: layer[-1] trả về tensor thuần
        def hook_ffn(module, input, output):
            return self.adapter_ffn(output)

        self.block.layer[0].register_forward_hook(hook_attn)
        self.block.layer[-1].register_forward_hook(hook_ffn)

    def forward(self, *args, **kwargs):
        # Chuyển thẳng mọi tham số vào T5Block gốc
        # Hook sẽ tự động chèn adapter vào đúng vị trí
        return self.block(*args, **kwargs)


Model chính với Adapter

In [4]:
from transformers import T5EncoderModel, T5Config

class T5WithAdapter(nn.Module):
    def __init__(self, model_name: str, num_labels: int, adapter_dim: int):
        super().__init__()
        self.config = T5Config.from_pretrained(model_name)
        self.t5 = T5EncoderModel.from_pretrained(model_name)
        hidden_dim = self.config.d_model 

        # Chèn Adapter vào từng block của Encoder
        for i, block in enumerate(self.t5.encoder.block):
            self.t5.encoder.block[i] = T5BlockWithAdapter(block, hidden_dim, adapter_dim)

        self.classifier = nn.Linear(hidden_dim, num_labels)
        
        self._freeze_backbone()
        self._count_params()

    def _freeze_backbone(self):
        for param in self.t5.parameters():
            param.requires_grad = False
        for block in self.t5.encoder.block:
            for param in block.adapter_attn.parameters(): param.requires_grad = True
            for param in block.adapter_ffn.parameters(): param.requires_grad = True
        for param in self.classifier.parameters(): param.requires_grad = True

    def _count_params(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Total params: {total:,}")
        print(f"Trainable params: {trainable:,} ({trainable/total*100:.2f}%)")

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.t5(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state.mean(dim=1)
        logits = self.classifier(pooled)
        loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return {"loss": loss, "logits": logits}

In [5]:
def load_qnli_data(tokenizer, max_length: int, batch_size: int):
    """Load GLUE QNLI và tokenize."""
    dataset = load_dataset("glue", "qnli")

    def preprocess(examples):
        return tokenizer(
            examples["question"],
            examples["sentence"],
            max_length=max_length,
            truncation=True,
            padding="max_length",
        )

    tokenized = dataset.map(preprocess, batched=True,
                            remove_columns=["question", "sentence", "idx"])
    tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

    train_loader = DataLoader(tokenized["train"],
                              batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(tokenized["validation"],
                              batch_size=batch_size, shuffle=False)

    print(f"Train size: {len(tokenized['train']):,}")
    print(f"Val size:   {len(tokenized['validation']):,}")
    return train_loader, val_loader

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
train_loader, val_loader = load_qnli_data(
    tokenizer, CONFIG["max_length"], CONFIG["batch_size"]
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Generating test split:   0%|          | 0/5463 [00:00<?, ? examples/s]

Map:   0%|          | 0/104743 [00:00<?, ? examples/s]

Map:   0%|          | 0/5463 [00:00<?, ? examples/s]

Map:   0%|          | 0/5463 [00:00<?, ? examples/s]

Train size: 104,743
Val size:   5,463


Training loop

In [6]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, total_correct, total_samples = 0, 0, 0

    for batch in tqdm(loader, desc="Training", leave=False):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask, labels=labels)
        loss = outputs["loss"]
        loss.backward()

        # Gradient clipping
        nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], max_norm=1.0
        )
        optimizer.step()
        scheduler.step()

        preds = outputs["logits"].argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_loss    += loss.item() * len(labels)
        total_samples += len(labels)

    return total_loss / total_samples, total_correct / total_samples


@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    metric = evaluate.load("glue", "qnli")

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        outputs = model(input_ids, attention_mask)
        preds = outputs["logits"].argmax(dim=-1)
        metric.add_batch(predictions=preds.cpu(), references=labels.cpu())

    return metric.compute()


def run_training():
    # Khởi tạo model
    model = RobertaWithAdapter(
        CONFIG["model_name"],
        CONFIG["num_labels"],
        CONFIG["adapter_dim"]
    ).to(device)

    # Chỉ optimize các param trainable
    optimizer = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"]
    )

    total_steps   = len(train_loader) * CONFIG["num_epochs"]
    warmup_steps  = int(total_steps * CONFIG["warmup_ratio"])
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    best_acc = 0
    history = []

    for epoch in range(1, CONFIG["num_epochs"] + 1):
        print(f"\n{'='*50}")
        print(f"Epoch {epoch}/{CONFIG['num_epochs']}")

        train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, scheduler, device
        )
        val_metrics = evaluate_model(model, val_loader, device)
        val_acc = val_metrics["accuracy"]

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Accuracy: {val_acc:.4f}")

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_acc": val_acc
        })

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_adapter_qnli.pt")
            print(f"✓ Saved best model (acc={best_acc:.4f})")

    print(f"\nBest Val Accuracy: {best_acc:.4f}")
    return model, history

model, history = run_training()

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total params:     127,026,434
Trainable params: 2,380,802  (1.87%)
Frozen params:    124,645,632

Epoch 1/5


Train Loss: 0.4535 | Train Acc: 0.7787
Val Accuracy: 0.8823
✓ Saved best model (acc=0.8823)

Epoch 2/5


Train Loss: 0.3201 | Train Acc: 0.8678
Val Accuracy: 0.9001
✓ Saved best model (acc=0.9001)

Epoch 3/5


Train Loss: 0.2827 | Train Acc: 0.8855
Val Accuracy: 0.9123
✓ Saved best model (acc=0.9123)

Epoch 4/5


Train Loss: 0.2590 | Train Acc: 0.8958
Val Accuracy: 0.9143
✓ Saved best model (acc=0.9143)

Epoch 5/5


Train Loss: 0.2414 | Train Acc: 0.9040
Val Accuracy: 0.9162
✓ Saved best model (acc=0.9162)

Best Val Accuracy: 0.9162


In [7]:
# Kết quả trong paper (Table 1): AP đạt 91.58 trên QNLI với T5-base
# RoBERTa-base của chúng ta dự kiến ~91-93%

print("\n=== Kết quả so với paper ===")
print(f"Paper (AP, T5-base):    91.58")
print(f"Của bạn (adapter_dim={CONFIG['adapter_dim']}): {max(h['val_acc'] for h in history)*100:.2f}")

# In bảng history
print("\nEpoch | Train Loss | Train Acc | Val Acc")
print("-" * 45)
for h in history:
    print(f"  {h['epoch']}   |   {h['train_loss']:.4f}   |   {h['train_acc']:.4f}  |  {h['val_acc']:.4f}")


=== Kết quả so với paper ===
Paper (AP, T5-base):    91.58
Của bạn (adapter_dim=64): 91.62

Epoch | Train Loss | Train Acc | Val Acc
---------------------------------------------
  1   |   0.4535   |   0.7787  |  0.8823
  2   |   0.3201   |   0.8678  |  0.9001
  3   |   0.2827   |   0.8855  |  0.9123
  4   |   0.2590   |   0.8958  |  0.9143
  5   |   0.2414   |   0.9040  |  0.9162
